# Thumbnail Generation Notebook

## Parameters

In [12]:
# --- Parameters ---

MAX_WIDTH         = 150        # int or None — downscale if image width exceeds this
MAX_HEIGHT        = None        # int or None — downscale if image height exceeds this
COMPRESSION_QUALITY = 70       # 0–100 (WebP quality)
INPUT_DIRECTORY   = "full"        # relative to this notebook
OUTPUT_DIRECTORY  = "tiny"   # relative to this notebook
INCLUDE_ALPHA     = False      # keep transparency channel (RGBA) if True

## Generate Thumbnails

In [13]:
import os
from pathlib import Path
from PIL import Image

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".tif", ".webp"}

notebook_dir = Path().resolve()
input_dir    = (notebook_dir / INPUT_DIRECTORY).resolve()
output_dir   = (notebook_dir / OUTPUT_DIRECTORY).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

def compute_size(original: tuple[int, int]) -> tuple[int, int]:
    """Return scaled (width, height) respecting MAX_WIDTH / MAX_HEIGHT, maintaining aspect ratio."""
    w, h = original
    if MAX_WIDTH is not None and w > MAX_WIDTH:
        h = round(h * MAX_WIDTH / w)
        w = MAX_WIDTH
    if MAX_HEIGHT is not None and h > MAX_HEIGHT:
        w = round(w * MAX_HEIGHT / h)
        h = MAX_HEIGHT
    return w, h

results = []  # [(input_path, output_path, input_bytes, output_bytes)]


to_update = 'UKLG - HS'

input_files = sorted(
    p for p in input_dir.iterdir()
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS and to_update in p.stem
)

if not input_files:
    print(f"No supported image files found in: {input_dir}")
else:
    for src in input_files:
        dst = output_dir / (src.stem + ".webp")

        with Image.open(src) as img:
            # Resolve colour mode
            if INCLUDE_ALPHA:
                img = img.convert("RGBA")
            else:
                img = img.convert("RGB")

            new_size = compute_size(img.size)
            if new_size != img.size:
                img = img.resize(new_size, Image.LANCZOS)

            img.save(dst, format="WEBP", quality=COMPRESSION_QUALITY)

        input_bytes  = src.stat().st_size
        output_bytes = dst.stat().st_size
        results.append((src, dst, input_bytes, output_bytes))
        print(f"  {src.name[:30].ljust(30)}  →  {dst.name[:30].ljust(30)}  ({new_size[0]}×{new_size[1]})")

print(f"\nDone — {len(results)} image(s) written to: {output_dir}")

  UKLG - HS1 - Rocannon's World.  →  UKLG - HS1 - Rocannon's World.  (150×244)
  UKLG - HS2 - Planet of Exile.j  →  UKLG - HS2 - Planet of Exile.w  (150×230)
  UKLG - HS3 - City of Illusions  →  UKLG - HS3 - City of Illusions  (150×230)
  UKLG - HS4 - The Left Hand of   →  UKLG - HS4 - The Left Hand of   (150×222)
  UKLG - HS5 - The Word for Worl  →  UKLG - HS5 - The Word for Worl  (150×244)

Done — 5 image(s) written to: C:\Users\Lewis\Documents\GitHub\NE2\nervous-energy\public\books\covers\tiny


## Size Comparison

In [14]:
def fmt(n: int) -> str:
    """Human-readable file size."""
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"

col_w = 30

if results:
    if not col_w:
        col_w = max(len(r[0].name) for r in results)
    header = f"{'File':<{col_w}}   {'Input':>10}   {'Output':>10}   {'Saved':>10}   {'Ratio':>7}"
    print(header)
    print("-" * len(header))

    total_in = total_out = 0
    for src, dst, in_b, out_b in results:
        saved = in_b - out_b
        ratio = out_b / in_b if in_b else 0
        print(f"{src.name[:col_w]:<{col_w}}   {fmt(in_b):>10}   {fmt(out_b):>10}   {fmt(saved):>10}   {ratio:>6.1%}")
        total_in  += in_b
        total_out += out_b

    print("-" * len(header))
    total_saved = total_in - total_out
    total_ratio = total_out / total_in if total_in else 0
    print(f"{'TOTAL':<{col_w}}   {fmt(total_in):>10}   {fmt(total_out):>10}   {fmt(total_saved):>10}   {total_ratio:>6.1%}")

File                                  Input       Output        Saved     Ratio
-------------------------------------------------------------------------------
UKLG - HS1 - Rocannon's World.     269.5 KB       6.1 KB     263.4 KB     2.2%
UKLG - HS2 - Planet of Exile.j     289.5 KB       7.4 KB     282.1 KB     2.6%
UKLG - HS3 - City of Illusions       2.7 MB       8.0 KB       2.7 MB     0.3%
UKLG - HS4 - The Left Hand of      443.6 KB       7.4 KB     436.1 KB     1.7%
UKLG - HS5 - The Word for Worl       4.8 MB       7.7 KB       4.8 MB     0.2%
-------------------------------------------------------------------------------
TOTAL                                8.6 MB      36.6 KB       8.5 MB     0.4%
